# Bước 2: probe ViT-g/384 của Meta trên EK100 (video gốc) + chấm lại logits

Mục đích: kiểm tra cách chỉnh logits (logit adjustment và ghép verb–noun, bước 0d) có tăng điểm trên **model khác** không. Nếu có,
bài báo có luận điểm "cách chỉnh này áp dụng được cho nhiều model". Notebook cũng kiểm chứng luôn con số của ViT-g/384 trong bài báo
(**39.7** action R@5).

**Chi phí:** ViT-g (1 tỷ tham số, 384px, 9216 token mỗi clip) nặng hơn ViT-L khoảng 8–10 lần. Ước tính khoảng 0.14 clip/s trên 2×T4,
tức khoảng 18 giờ GPU cho toàn bộ val, chỉ tính giao thức `official`. Vì vậy val được **chia làm 2 phần theo người tham gia**:
- `PART = 0` chạy trên một tài khoản, `PART = 1` chạy trên tài khoản khác, **song song**, mỗi phần khoảng 9–10 giờ;
- khi chạy xong cả hai: mở lại notebook của một phần, *Add Input* output của phần kia, rồi Save & Run All. Cell cuối sẽ ghép 2 phần và
  tính điểm trên toàn bộ val (phần eval đã xong sẽ được bỏ qua).

Nếu một phần chưa xong trong 12 giờ: *Add Input* output của chính phần đó rồi chạy lại, notebook sẽ chạy tiếp từ chỗ dừng.

**Cài đặt:** GPU **T4 x2**, Internet **On**. Không cần dataset video, vì notebook tải video gốc rồi xoá ngay sau khi eval.
Sau khoảng 30 phút, xem tốc độ trong log `eval_gpu*.log` (dòng `clip/s`). Nếu chậm hơn nhiều so với ước tính, gửi tôi log đó.

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"  # version the code was checked against
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /kaggle/working/vjepa2 && cd /kaggle/working/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops
import os
os.environ["VJEPA2_ROOT"] = "/kaggle/working/vjepa2"
R = "/kaggle/working/repo"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq install -y aria2 > /dev/null 2>&1 && aria2c --version | head -1
!pip install -q av


In [ ]:
# ViT-g/384 checkpoint is large: download to /kaggle/tmp, keep only target encoder + predictor in fp16, delete the rest
import os
CK = "/kaggle/tmp/ckpt"
os.makedirs(CK, exist_ok=True)
if not os.path.exists(f"{CK}/vitg-384-slim.pt"):
    !wget -q -nc -P $CK https://dl.fbaipublicfiles.com/vjepa2/vitg-384.pt
    !ls -lh $CK
    !python $R/scripts/slim_checkpoint.py $CK/vitg-384.pt $CK/vitg-384-slim.pt && rm -f $CK/vitg-384.pt
!wget -q -nc -P $CK https://dl.fbaipublicfiles.com/vjepa2/evals/ek100-vitg-384.pt
!ls -lh $CK; df -h /kaggle/tmp | tail -1


In [ ]:
# Official annotations (the full train csv is required: probe output classes are derived from it)
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!ls /kaggle/input; find /kaggle/input -maxdepth 3 | head -30

In [ ]:
# ---- settings ----
PART = 0          # 0 on one account, 1 on another
NUM_PARTS = 2
ANCHORS = "official"   # "official action_start" doubles the GPU time
# ------------------
import glob, os, shutil
OUT = f"/kaggle/working/logits_vitg_part{PART}"
prev = glob.glob(f"/kaggle/input/**/logits_vitg_part{PART}", recursive=True)
if prev:
    shutil.copytree(prev[0], OUT, dirs_exist_ok=True); print("resumed from", prev[0])
else:
    print("fresh run of part", PART)
!python $R/scripts/prepare_ek100.py --download_layout --video_root /kaggle/tmp/ek100 \
    --train_csv /kaggle/working/ek100-ann/EPIC_100_train.csv \
    --val_csv /kaggle/working/ek100-ann/EPIC_100_validation.csv \
    --video_info /kaggle/working/ek100-ann/EPIC_100_video_info.csv \
    --out_dir /kaggle/working/work | grep -E '"val_clips"'
# participant-disjoint parts, balanced by clip count (same split rule as the cross-fitting folds)
import sys, pandas as pd
sys.path.insert(0, f"{R}/scripts")
from cv_probe import participant_folds
clips = pd.read_csv("/kaggle/working/work/val_clips.csv")
clips["participant"] = clips.video_id.str.split("_").str[0]
clips["part"] = participant_folds(clips, NUM_PARTS)
mine = clips[clips.part == PART]
VIDEOS = f"/kaggle/working/vitg_part{PART}_videos.txt"
open(VIDEOS, "w").write("\n".join(sorted(mine.video_id.unique())))
print(f"part {PART}: {mine.video_id.nunique()} videos, {len(mine)} clips, participants {sorted(mine.participant.unique())}")


In [ ]:
!python $R/scripts/stream_eval_ek100.py --work_dir /kaggle/working/work \
    --video_info /kaggle/working/ek100-ann/EPIC_100_video_info.csv --out_dir $OUT --videos_file $VIDEOS \
    --vitl_ckpt $CK/vitg-384-slim.pt --probe_ckpt $CK/ek100-vitg-384.pt \
    --gpus 0 1 --anchors $ANCHORS --dtype fp16 --decode_fallback pyav \
    --eval_args "--config $R/configs/ek100_vitg384_inference.yaml" \
    --parallel_downloads 2 --connections 8 --max_videos_on_disk 6 --max_hours 10.5
!grep -h "clip/s" $OUT/eval_gpu*.log | tail -n 2


In [ ]:
# Merge every part available (this one + other parts attached with Add Input), then score + re-score
import glob, os, shutil, subprocess
parts = {PART: OUT}
for k in range(NUM_PARTS):
    if k != PART:
        found = glob.glob(f"/kaggle/input/**/logits_vitg_part{k}", recursive=True)
        if found:
            parts[k] = found[0]
print("parts:", parts)
M = "/kaggle/working/merged_vitg"
shutil.rmtree(M, ignore_errors=True)
for k, d in parts.items():
    for sd in glob.glob(f"{d}/*/shard*"):
        anchor, shard = sd.split("/")[-2], int(sd.split("shard")[-1])
        shutil.copytree(sd, f"{M}/{anchor}/shard{10 * k + shard}")
for anchor in ANCHORS.split():
    cmd = (f"python {R}/scripts/compute_metrics.py --work_dir /kaggle/working/work --logits_dir {M}/{anchor} "
           f"--paper_action 39.7 --out /kaggle/working/vitg_{anchor}.json")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"[{anchor}] PARTIAL (not all parts done):", (r.stderr or r.stdout).strip()[-200:])
        r = subprocess.run(cmd + " --allow_partial", shell=True, capture_output=True, text=True)
    print(f"== ViT-g/384 {anchor} ==\n" + r.stdout)
    !python $R/scripts/posthoc_ek100.py --work_dir /kaggle/working/work \
        --train_csv /kaggle/working/ek100-ann/EPIC_100_train.csv --logits_dir {M}/{anchor} \
        --out /kaggle/working/posthoc_vitg_{anchor}.json


Gửi lại output của cell cuối (điểm ViT-g/384 và bảng chấm lại). Nếu mới xong một phần, các con số sẽ được đánh dấu PARTIAL.